# Density based clustering methods

In this practice you will:
- implement the clustering method presented in the paper: Rodriguez, Alex, and Alessandro Laio. “Clustering by Fast Search and Find of Density Peaks.” Science 344, no. 6191 (2014): 1492–96. https://doi.org/10.1126/science.1242072.
- apply it to the MNIST dataset
- apply the DBSCAN method on the MNIST dataset
- learn how clustering performances with these two methods are affected by the parameters of the algorithms

## 0. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

## 1. Create a synthetic dataset

In [ ]:
mean1 = [1, 6]
cov1 = [[0.5, 0.0],
        [0.0, 0.5]]
X1 = np.random.multivariate_normal(mean1, cov1, 120)
mean2 = [9, 7]
cov2 = [[2.5, 1.8],
        [1.8, 1.5]]
X2 = np.random.multivariate_normal(mean2, cov2, 180)
mean3 = [-4, 4]
cov3 = [[4.2, -2.4],
        [-2.4, 4.0]]
X3 = np.random.multivariate_normal(mean3, cov3, 100)
X = np.vstack([X1, X2, X3])
y_true = np.array([0]*len(X1) + [1]*len(X2) + [2]*len(X3))
feature_names = ['feat_1', 'feat_2']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pd.DataFrame(X_scaled, columns=feature_names).describe().loc[["mean", "std"]]
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    s=25,
    alpha=0.7
)
plt.xlabel(feature_names[0] + " (scaled)")
plt.ylabel(feature_names[1] + " (scaled)")
plt.title("Synthetic dataset")
plt.show()

## 2. Clustering based on density peaks

Write a function to calculate for each point in the dataset:
* the local density, **rho**, defined as the number of samples within a given distance epsilon;
* the index of the closest sample with higher density, **inds_closest_higher_rho**. For the sample with highest density set this value to -1;
* the distance to the closest sample with higher density, **delta**.

The function should returns the arrays rho, inds_closest_higher_rho, and delta, and it should plot delta as a function of rho. Test the function using the synthetic dataset, and values of epsilon equal to 0.1, 1, and 10. How does the plot change ? What is the optimal value ?

In [ ]:
def density_peaks(X, eps):
    D = pairwise_distances(X)
    rho = np.sum(D < eps, axis = 1)
    inds_maxmin_rho = np.argsort(rho)[::-1]
    delta = np.zeros(len(rho)) # distance from the clostest point of higher density
    delta[inds_maxmin_rho[0]] = np.max(D) # inoitiale to max distance among points
    inds_closest_higher_rho = -1*np.ones(len(rho), dtype = int) # index of that point
    for ind in range(1, len(rho)):
        ind_sample = inds_maxmin_rho[ind]
        inds_higher_density = inds_maxmin_rho[:ind]
        inds_closest_higher_rho[ind_sample] = inds_higher_density[
            np.argmin(D[ind_sample, inds_higher_density])
        ]
        delta[ind_sample] = np.min(D[ind_sample, inds_higher_density])
    f = plt.figure()
    ax = f.add_subplot(1,1,1)
    ax.plot(rho, delta, 'o')
    return rho, delta, inds_closest_higher_rho

In [ ]:
r, d, inds = density_peaks(X, 1.0)

Write a function that takes as inputs the arrays produced by the previous function, and threshold values for rho, **min_rho**, and delta, **min_delta**. The function should:
* define as cluster centers all the points with rho > min_rho and delta > min_delta. I suggest to highlight the cluster centers in a plot of delta Vs rho, to check that the correct samples were selected;
* assign each point to the same cluster of its closest sample of higher density. To do this I suggest to:
  * assign a cluster label to the cluster centers
  * make a cycle over samples in decreasing order of density
      * if the sample does not have a cluster label already assigned to it, use the cluster label of its closest sample of higher density

The function should return the labels of the clusters. Apply the function to the synthetic data, and plot the data using different colors for the clusters. Define the values of min_rho and min_delta based on the plot produced by the previous function.

In [ ]:
def clustering_dp(rho, delta, inds_closest_higher_rho, min_rho, min_delta):
    inds_centers = np.logical_and(rho > min_rho, delta > min_delta)
    f = plt.figure()
    ax = f.add_subplot(1,1,1)
    ax.plot(rho, delta, 'o')
    ax.plot([min_rho, np.max(rho)], [min_delta, min_delta], 'k:')
    ax.plot([min_rho, min_rho], [min_delta, np.max(delta)], 'k:')
    ax.plot(rho[inds_centers], delta[inds_centers], 'ro')
    labels = -1*np.ones(len(rho), dtype = int)
    labels[np.where(inds_centers)[0]] = np.arange(np.sum(inds_centers))
    for ind_sample in np.argsort(rho)[::-1]:
        if labels[ind_sample] == -1:
            labels[ind_sample] = labels[inds_closest_higher_rho[ind_sample]]
    return labels

In [ ]:
clustering_dp(r, d, inds, 10.0, 3.0)

In [ ]:
labels = clustering_dp(r, d, inds, 10.0, 3.0)
f = plt.figure()
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    s=25,
    c=labels,
    alpha=0.7
)
plt.xlabel(feature_names[0] + " (scaled)")
plt.ylabel(feature_names[1] + " (scaled)")
plt.title("Synthetic biomedical dataset")
plt.show()

## 3. Clustering of the MNIST dataset based on density peaks

The MNIST dataset includes images of hand-written digits. Each image is 8x8. The block of code below load the data, **X**, the correct labels, **y**, and it plots a random image from the dataset.

* Use the functions defined before to identify possible candidates as cluster centers.Modify the value of epsilon to get a number of isolated density peaks around 10-15
* Perform the clustering
* Compute the adjusted rank index of the clustering results
* Plots some examples of images for all the clusters identified

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
images = datasets.load_digits()
X = images['data']
n_images = X.shape[0]
y = images['target']
print('Number of images:', n_images)
X_images = X.reshape(n_images, 8, 8)
ind = np.random.randint(n_images)
plt.imshow(X_images[ind], cmap='grey')
plt.title(y[ind])

Number of images: 1797


NameError: name 'plt' is not defined

In [ ]:
# this block of code imports a different dataset with images at higher resolution. I suggest to use the previous one,
# and then maybe explore this one. In case remember that you need to adjust the parameters of the algoriths,
# as the average distance between samples would be significanly different for these images
from sklearn.datasets import fetch_openml
n_images = 1000 # I downsample the original dataset to this number
X,y = fetch_openml('mnist_784', return_X_y=True)
inds = np.random.choice(np.arange(len(y)).astype(int), n_images)
X = X.to_numpy()
y = y.to_numpy()
X = X[inds,:]
y = y[inds]
n_images = X.shape[0]
X_images = X.reshape(n_images, 28, 28)
X = X_images.reshape(n_images, 28 * 28)
print('Number of images:', n_images)
ind = np.random.randint(n_images)
plt.imshow(X_images[ind], cmap='grey')
plt.title(y[ind])

In [ ]:
# 1. Identificare i picchi di densità (modifica eps per ottenere ~10-15 picchi)
# Nota: per il dataset digits 8x8, un eps intorno a 28 spesso restituisce buoni risultati
eps_mnist = 28.0
r_mnist, d_mnist, inds_mnist = density_peaks(X, eps_mnist)

# 2. Eseguire il clustering
# I valori di min_rho e min_delta vanno scelti osservando il grafico generato dalla funzione precedente.
min_rho_mnist = 20.0
min_delta_mnist = 20.0
labels_dp = clustering_dp(r_mnist, d_mnist, inds_mnist, min_rho_mnist, min_delta_mnist)

# 3. Calcolare l'Adjusted Rand Index (ARI)
ari_dp = adjusted_rand_score(y, labels_dp)
print(f"Adjusted Rand Index (Density Peaks): {ari_dp:.3f}")

# 4. Visualizzare alcuni esempi di immagini per tutti i cluster identificati
n_clusters_dp = np.max(labels_dp) + 1
max_plots = 5

for i_cluster in range(n_clusters_dp):
    # Trova gli indici delle immagini che appartengono a questo cluster
    cluster_indices = np.where(labels_dp == i_cluster)[0]

    # Crea una figura per il cluster
    f = plt.figure(figsize=(10, 2))
    f.suptitle(f"Cluster {i_cluster} (Size: {len(cluster_indices)})")

    # Plotta fino a 'max_plots' immagini
    for i_plot in range(min(max_plots, len(cluster_indices))):
        ax = f.add_subplot(1, max_plots, i_plot + 1)
        ax.imshow(X_images[cluster_indices[i_plot]], cmap='grey')
        ax.axis('off')

    plt.show()

## 4. Clustering of the MNIST dataset using DBSCAN

Clusterize the same data using the DBSCAN algorithm. Set epsilon to 20 and the minimum number of samples to 10. Compute the adjusted rank index excluding the samples labelled ad noise by DBSCAN.

In [ ]:
from sklearn.cluster import DBSCAN
model = DBSCAN(eps=20, min_samples=10)
model.fit(X)
clusters = model.labels_
n_clusters = np.max(clusters) + 1
noise_samples = np.sum(clusters == -1)
print("Number of clusters:", n_clusters)
print("Number of samples classified as noise:", noise_samples)

In [ ]:
model = DBSCAN(eps=20, min_samples=10)
model.fit(X)
clusters_dbscan = model.labels_

n_clusters_dbscan = np.max(clusters_dbscan) + 1
noise_samples = np.sum(clusters_dbscan == -1)

print("Number of clusters:", n_clusters_dbscan)
print("Number of samples classified as noise:", noise_samples)

# Calcolo dell'Adjusted Rand Index escludendo i sample classificati come rumore (-1)
mask_no_noise = clusters_dbscan != -1
y_filtered = y[mask_no_noise]
clusters_filtered = clusters_dbscan[mask_no_noise]

ari_dbscan = adjusted_rand_score(y_filtered, clusters_filtered)
print(f"Adjusted Rand Index (DBSCAN, escludendo il rumore): {ari_dbscan:.3f}")

# Visualizzazione dei cluster trovati da DBSCAN
max_plots = 5

for i_cluster in range(n_clusters_dbscan):
    cluster_indices = np.where(clusters_dbscan == i_cluster)[0]

    f = plt.figure(figsize=(10, 2))
    f.suptitle(f"DBSCAN Cluster {i_cluster} (Size: {len(cluster_indices)})")

    for i_plot in range(min(max_plots, len(cluster_indices))):
        ax = f.add_subplot(1, max_plots, i_plot + 1)
        ax.imshow(X_images[cluster_indices[i_plot]], cmap='grey')
        ax.axis('off')

    plt.show()

In [ ]:
for i_cluster in range(n_clusters):
    f = plt.figure()
    i_plot +=1
    ax = f.add_subplot(1,max_plots, i_plots)
    ax.imshow(X_images[i_image])
    if i_plot == max_plots:
      break